
```
docker run -it --rm \
    --name aic_eval \
    --network host \
    --gpus all \
    -e DISPLAY=$DISPLAY \
    -v /tmp/.X11-unix:/tmp/.X11-unix \
    ghcr.io/intrinsic-dev/aic/aic_eval:latest \
    ground_truth:=false start_aic_engine:=true
```
   
   
---   
   Here's the breakdown:

`docker run -it --rm`
   - `docker run` - create and start a new container
   - `-i` (interactive) - keep STDIN open so you can send input
   - `-t` (tty) - allocate a pseudo-terminal (gives you a proper terminal with
     colors, Ctrl+C, etc.)
   - `--rm` - automatically delete the container when it stops (clean it after
     yourself)


`--name aic_eval`     
   Names the container `aic_eval` so you can reference it by name (e.g, 
   `docker exec aic_eval ..., docker stop aic_eval`) instead of a random hex ID.


`--network host`   
   Instead of Docker's default isolated virtual network, the container shares 
   the host's (WSL2's) network stack directly. This means:
      * Container sees the same IP, ports, interfaces as WSL2
      * ROS 2 Zenoh middleware can communicate freely between the container and
        your `pixi` policy running outside
      * This is why it fixed the DNS/networking issue -- no NAT/bridge layer
        to break things.


`--gpus all`          
   Passes all NVIDIA GPUs to the container via the NVIDIA Container Toolkit. 
   Without this, Gazebo would run CPU-only rendering (slow, ugly). With it, the
   RTX 5090 handles physics and rendering.


`-e DISPLAY=$DISPLAY`
   Sets the `DISPLAY` environment variable inside the container to match your
   WSL2 host's value. This tells GUI apps (Gazebo, RViz) where to draw their 
   windows. In WSL2 with WSLg, this is typically `:0`.


`-v /tmp/.X11-unix:/tmp/.X11-unix`
   Bind-mounts the X11 socket directory from WSL2 into the container. The
   `DISPLAY` variable tells apps which display, and this mount gives them access
   to the actual Unix socket file that connects to the WSLg display server.
   Together, these two flags enable GUI windows from inside the container to
   appear on your Windows desktop.   




---